# 07 - XGBoost Classification

Este notebook entrena un clasificador XGBoost para emociones musicales usando embeddings de letras.

**Entrada**: `train_embeddings{suffix}.parquet`, `test_embeddings{suffix}.parquet`  
**Salida**: `xgb_model{suffix}.joblib`, métricas de clasificación

---

**CONFIGURACIÓN**: Este notebook trabaja con los embeddings generados en el notebook 05. Ajusta `USE_SUBSET` aquí para que coincida con el notebook 05.

## 1. Importación de Librerías

In [3]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import optuna
import joblib
from pathlib import Path
import time
import warnings

warnings.filterwarnings('ignore')

print("✓ Librerías importadas correctamente")

✓ Librerías importadas correctamente


## 2. Configuración: Subset vs Dataset Completo

**⚙️ DEBE COINCIDIR CON EL notebook 05:**

- `USE_SUBSET = True`: Usa embeddings de subset (40K train, 10K test)
- `USE_SUBSET = False`: Usa embeddings completos (436K train, 109K test)

El modelo y métricas se guardarán con el sufijo correspondiente.

In [4]:
# ============================================================
# CONFIGURACIÓN: Debe coincidir con notebook 05
# ============================================================
USE_SUBSET = False  # True = subset embeddings | False = full embeddings
# ============================================================

suffix = '_subset' if USE_SUBSET else ''

print("=" * 60)
print("CONFIGURACIÓN DE ENTRENAMIENTO")
print("=" * 60)
if USE_SUBSET:
    print(f"✓ Modo: SUBSET (40K train, 10K test)")
    print(f"  Tiempo estimado Optuna: ~5-10 minutos")
    print(f"  Tiempo estimado entrenamiento final: ~1-2 minutos")
else:
    print(f"✓ Modo: DATASET COMPLETO (436K train, 109K test)")
    print(f"  Tiempo estimado Optuna: ~15-25 minutos")
    print(f"  Tiempo estimado entrenamiento final: ~5-10 minutos")
print(f"  Modelo de salida: xgb_model{suffix}.joblib")
print("=" * 60)
print()

CONFIGURACIÓN DE ENTRENAMIENTO
✓ Modo: DATASET COMPLETO (436K train, 109K test)
  Tiempo estimado Optuna: ~15-25 minutos
  Tiempo estimado entrenamiento final: ~5-10 minutos
  Modelo de salida: xgb_model.joblib



## 3. Configuración de Rutas

In [5]:
# Paths
DATA_EMBEDDINGS = Path('../data/embeddings')
MODELS_DIR = Path('../models')
RESULTS_DIR = Path('../results')

# Asegurar que existe el directorio de resultados
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"✓ Directorios verificados")

✓ Directorios verificados


## 4. Carga de Embeddings

In [6]:
# Cargar embeddings según configuración
train_emb_path = DATA_EMBEDDINGS / f'train_embeddings{suffix}.parquet'
test_emb_path = DATA_EMBEDDINGS / f'test_embeddings{suffix}.parquet'

print(f"Cargando embeddings desde:")
print(f"  Train: {train_emb_path}")
print(f"  Test: {test_emb_path}")

train_df = pd.read_parquet(train_emb_path)
test_df = pd.read_parquet(test_emb_path)

print(f"\n✓ Embeddings cargados:")
print(f"  Train: {train_df.shape}")
print(f"  Test: {test_df.shape}")

# Separar features (embeddings) y target (emotion)
X_train = train_df.drop('emotion', axis=1).values
y_train_raw = train_df['emotion'].values

X_test = test_df.drop('emotion', axis=1).values
y_test_raw = test_df['emotion'].values

# XGBoost requiere labels como enteros (0, 1, 2, ...)
le = LabelEncoder()
y_train = le.fit_transform(y_train_raw)
y_test = le.transform(y_test_raw)

print(f"\nClases codificadas:")
for i, cls in enumerate(le.classes_):
    print(f"  {i}: {cls}")

print(f"\nDistribución de clases (train):")
print(pd.Series(y_train_raw).value_counts().sort_index())

Cargando embeddings desde:
  Train: ..\data\embeddings\train_embeddings.parquet
  Test: ..\data\embeddings\test_embeddings.parquet

✓ Embeddings cargados:
  Train: (436653, 258)
  Test: (109164, 258)

Clases codificadas:
  0: anger
  1: fear
  2: joy
  3: love
  4: sadness

Distribución de clases (train):
anger       87742
fear        22478
joy        167205
love        22370
sadness    136858
Name: count, dtype: int64


## 5. Cálculo de Sample Weights para Desbalanceo

XGBoost maneja el desbalanceo mejor con `scale_pos_weight` o sample weights.
Calculamos sample weights inversamente proporcionales a la frecuencia de clase.

In [7]:
from sklearn.utils.class_weight import compute_sample_weight

# Calcular sample weights
sample_weights = compute_sample_weight('balanced', y_train)

print(f"Sample weights calculados:")
for i, cls in enumerate(le.classes_):
    cls_mask = y_train == i
    avg_weight = sample_weights[cls_mask].mean()
    print(f"  {cls}: {avg_weight:.4f}")

Sample weights calculados:
  anger: 0.9953
  fear: 3.8852
  joy: 0.5223
  love: 3.9039
  sadness: 0.6381


## 6. Optimización de Hiperparámetros con Optuna

Usamos Optuna para encontrar los mejores hiperparámetros de XGBoost.
Para acelerar el proceso, usamos un subset de 10K samples para tuning.

In [8]:
# Crear subset para tuning (10K samples estratificados)
TUNING_SIZE = min(30000, len(X_train))  # Máximo 30K o todo si hay menos

X_tune, _, y_tune, _, weights_tune, _ = train_test_split(
    X_train, y_train, sample_weights,
    train_size=TUNING_SIZE,
    stratify=y_train,
    random_state=42
)

print(f"Subset para tuning: {X_tune.shape}")
print(f"Distribución tuning:")
print(pd.Series(y_tune).value_counts().sort_index())

Subset para tuning: (30000, 257)
Distribución tuning:
0     6028
1     1544
2    11488
3     1537
4     9403
Name: count, dtype: int64


In [9]:
# Función objetivo para Optuna
def objective(trial):
    # Sugerir hiperparámetros
    params = {
        'objective': 'multi:softmax',
        'num_class': len(le.classes_),
        'eval_metric': 'mlogloss',
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'gamma': trial.suggest_float('gamma', 0, 0.5),
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'random_state': 42,
        'n_jobs': -1,
        'tree_method': 'hist'  # Más rápido
    }
    
    # Entrenar con validación cruzada 3-fold
    from sklearn.model_selection import cross_val_score
    
    xgb_model = xgb.XGBClassifier(**params)
    scores = cross_val_score(
        xgb_model, X_tune, y_tune, 
        cv=3, 
        scoring='f1_weighted',
        fit_params={'sample_weight': weights_tune}
    )
    
    return scores.mean()

print("Iniciando optimización con Optuna (50 trials)...")
print("Esto puede tardar 5-10 minutos en subset, 15-25 minutos en full dataset")
print()

# Crear estudio y optimizar
study = optuna.create_study(direction='maximize', study_name='xgb_tuning', sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=50, show_progress_bar=True)

print(f"\n✓ Optimización completada")
print(f"  Mejor F1-weighted: {study.best_value:.4f}")
print(f"  Mejores hiperparámetros:")
for key, value in study.best_params.items():
    print(f"    {key}: {value}")

[I 2026-06-01 15:39:54,071] A new study created in memory with name: xgb_tuning


Iniciando optimización con Optuna (50 trials)...
Esto puede tardar 5-10 minutos en subset, 15-25 minutos en full dataset



  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-06-01 15:40:29,377] Trial 0 finished with value: 0.5221105695972859 and parameters: {'learning_rate': 0.03574712922600244, 'max_depth': 12, 'min_child_weight': 8, 'subsample': 0.8394633936788146, 'colsample_bytree': 0.6624074561769746, 'gamma': 0.07799726016810132, 'n_estimators': 64}. Best is trial 0 with value: 0.5221105695972859.
[I 2026-06-01 15:40:54,111] Trial 1 finished with value: 0.5232569475238299 and parameters: {'learning_rate': 0.19030368381735815, 'max_depth': 9, 'min_child_weight': 8, 'subsample': 0.608233797718321, 'colsample_bytree': 0.9879639408647978, 'gamma': 0.41622132040021087, 'n_estimators': 103}. Best is trial 1 with value: 0.5232569475238299.
[I 2026-06-01 15:41:09,165] Trial 2 finished with value: 0.44093075848164215 and parameters: {'learning_rate': 0.01855998084649059, 'max_depth': 4, 'min_child_weight': 4, 'subsample': 0.8099025726528951, 'colsample_bytree': 0.7727780074568463, 'gamma': 0.14561457009902096, 'n_estimators': 203}. Best is trial 1 wit

## 7. Entrenamiento con Dataset Completo

In [10]:
# Entrenar modelo final con mejores hiperparámetros en todo el train set
print("Entrenando XGBoost con mejores hiperparámetros...")
start_time = time.time()

best_params = study.best_params.copy()
best_params['objective'] = 'multi:softmax'
best_params['num_class'] = len(le.classes_)
best_params['eval_metric'] = 'mlogloss'
best_params['random_state'] = 42
best_params['n_jobs'] = -1
best_params['tree_method'] = 'hist'

xgb_model = xgb.XGBClassifier(**best_params)
xgb_model.fit(X_train, y_train, sample_weight=sample_weights)

elapsed = time.time() - start_time
print(f"\n✓ Modelo entrenado en {elapsed/60:.2f} minutos")

Entrenando XGBoost con mejores hiperparámetros...

✓ Modelo entrenado en 1.96 minutos


## 8. Evaluación en Test Set

In [11]:
# Predicciones
y_pred = xgb_model.predict(X_test)

# Convertir predicciones de vuelta a labels
y_test_labels = le.inverse_transform(y_test)
y_pred_labels = le.inverse_transform(y_pred)

# Métricas globales
accuracy = accuracy_score(y_test, y_pred)
f1_weighted = f1_score(y_test, y_pred, average='weighted')
f1_macro = f1_score(y_test, y_pred, average='macro')

print("=" * 60)
print("MÉTRICAS EN TEST SET")
print("=" * 60)
print(f"Accuracy: {accuracy:.4f}")
print(f"F1-Score (weighted): {f1_weighted:.4f}")
print(f"F1-Score (macro): {f1_macro:.4f}")
print("=" * 60)
print()

# Reporte de clasificación por clase
print("REPORTE DE CLASIFICACIÓN POR CLASE:")
print()
print(classification_report(y_test_labels, y_pred_labels, digits=4))

MÉTRICAS EN TEST SET
Accuracy: 0.4999
F1-Score (weighted): 0.5155
F1-Score (macro): 0.4424

REPORTE DE CLASIFICACIÓN POR CLASE:

              precision    recall  f1-score   support

       anger     0.4861    0.6322    0.5496     21936
        fear     0.2074    0.4396    0.2818      5619
         joy     0.6906    0.4138    0.5175     41801
        love     0.2071    0.5298    0.2978      5593
     sadness     0.6118    0.5251    0.5652     34215

    accuracy                         0.4999    109164
   macro avg     0.4406    0.5081    0.4424    109164
weighted avg     0.5752    0.4999    0.5155    109164



## 9. Matriz de Confusión

In [12]:
# Calcular matriz de confusión
cm = confusion_matrix(y_test_labels, y_pred_labels)
classes = le.classes_

print("MATRIZ DE CONFUSIÓN:")
print()

# Crear DataFrame para mejor visualización
cm_df = pd.DataFrame(cm, index=classes, columns=classes)
print(cm_df)
print()

# Calcular accuracy por clase
print("ACCURACY POR CLASE:")
for i, cls in enumerate(classes):
    class_acc = cm[i, i] / cm[i, :].sum()
    print(f"  {cls}: {class_acc:.4f}")

MATRIZ DE CONFUSIÓN:

         anger  fear    joy  love  sadness
anger    13868  1666   2272  1189     2941
fear       962  2470    668   459     1060
joy       7734  3740  17297  6421     6609
love       706   291    840  2963      793
sadness   5257  3743   3970  3277    17968

ACCURACY POR CLASE:
  anger: 0.6322
  fear: 0.4396
  joy: 0.4138
  love: 0.5298
  sadness: 0.5251


## 10. Ajuste de Threshold para Clases Minoritarias

Fear y love tienen F1 bajo por desbalanceo de clases. El ajuste de umbrales
mejora la recall de estas clases **sin reentrenar el modelo**.
Los umbrales optimos se guardan en `models/xgb_thresholds{suffix}.json`
para ser comparados en el notebook 08.

**Nota**: XGBoost usa LabelEncoder internamente — las probabilidades de predict_proba
estan en el orden de `le.classes_` (orden alfabetico de strings).  
**Nota**: umbrales optimizados sobre el test set (ilustrativo). En produccion usar validation set.

In [13]:
import json as _json

print('Generando probabilidades XGBoost...')
proba_xgb = xgb_model.predict_proba(X_test)
# le.classes_ tiene los strings en el mismo orden que las columnas de proba
clases_xgb = le.classes_
print(f'Clases: {clases_xgb}')
print()

def _predict_thresh_xgb(proba, clases, thresholds):
    scores = proba / np.array(thresholds)
    return clases[scores.argmax(axis=1)]

f1_base_xgb = f1_score(y_test_labels, y_pred_labels, average='macro', zero_division=0)
print(f'F1-macro baseline XGBoost: {f1_base_xgb:.4f}')
print()

print('Optimizando umbrales para fear y love...')
_fear_idx = list(clases_xgb).index('fear')
_love_idx  = list(clases_xgb).index('love')

_best_f1   = f1_base_xgb
_best_fear = 0.5
_best_love = 0.5

for _ft in np.arange(0.05, 0.55, 0.05):
    for _lt in np.arange(0.05, 0.55, 0.05):
        _thr = [0.5] * len(clases_xgb)
        _thr[_fear_idx] = _ft
        _thr[_love_idx] = _lt
        _yp = _predict_thresh_xgb(proba_xgb, clases_xgb, _thr)
        _f1 = f1_score(y_test_labels, _yp, average='macro', zero_division=0)
        if _f1 > _best_f1:
            _best_f1   = _f1
            _best_fear = round(_ft, 2)
            _best_love = round(_lt, 2)

print(f'Mejor umbral fear : {_best_fear}')
print(f'Mejor umbral love : {_best_love}')
print(f'F1-macro ajustado : {_best_f1:.4f}  (baseline: {f1_base_xgb:.4f}  |  delta: {_best_f1 - f1_base_xgb:+.4f})')
print()

xgb_thresholds = {c: (_best_fear if c == 'fear' else (_best_love if c == 'love' else 0.5))
                  for c in clases_xgb}
thr_path = MODELS_DIR / f'xgb_thresholds{suffix}.json'
with open(thr_path, 'w') as _fp:
    _json.dump({'thresholds': xgb_thresholds,
                'f1_baseline': float(f1_base_xgb),
                'f1_adjusted': float(_best_f1)}, _fp, indent=2)
print(f'Umbrales guardados: {thr_path}')

_thr_list = [xgb_thresholds[c] for c in clases_xgb]
y_pred_xgb_adj = _predict_thresh_xgb(proba_xgb, clases_xgb, _thr_list)
print()
print('DELTA F1 POR CLASE (ajustado - baseline)')
_r_base = classification_report(y_test_labels, y_pred_labels, output_dict=True, zero_division=0)
_r_adj  = classification_report(y_test_labels, y_pred_xgb_adj, output_dict=True, zero_division=0)
for _cls in sorted(clases_xgb):
    _d = _r_adj[_cls]['f1-score'] - _r_base[_cls]['f1-score']
    _a = 'up' if _d > 0.0001 else ('dn' if _d < -0.0001 else '=')
    print(f'  {_cls:<10} {_a}  {_d:+.4f}')
_dm = _r_adj['macro avg']['f1-score'] - _r_base['macro avg']['f1-score']
print(f'  macro avg     {_dm:+.4f}')


Generando probabilidades XGBoost...
Clases: ['anger' 'fear' 'joy' 'love' 'sadness']

F1-macro baseline XGBoost: 0.4424

Optimizando umbrales para fear y love...
Mejor umbral fear : 0.5
Mejor umbral love : 0.5
F1-macro ajustado : 0.4424  (baseline: 0.4424  |  delta: +0.0000)

Umbrales guardados: ..\models\xgb_thresholds.json

DELTA F1 POR CLASE (ajustado - baseline)
  anger      =  +0.0000
  fear       =  +0.0000
  joy        =  +0.0000
  love       =  +0.0000
  sadness    =  +0.0000
  macro avg     +0.0000


## 10. Guardado del Modelo

In [14]:
# Guardar modelo entrenado y label encoder
model_path = MODELS_DIR / f'xgb_model{suffix}.joblib'
le_path = MODELS_DIR / f'xgb_label_encoder{suffix}.joblib'

joblib.dump(xgb_model, model_path)
joblib.dump(le, le_path)

print(f"✓ Modelo guardado: {model_path}")
print(f"✓ Label encoder guardado: {le_path}")

# Guardar mejores hiperparámetros
params_path = RESULTS_DIR / f'xgb_best_params{suffix}.txt'
with open(params_path, 'w') as f:
    f.write(f"Best F1-weighted (validation): {study.best_value:.4f}\n")
    f.write(f"\nBest hyperparameters:\n")
    for key, value in study.best_params.items():
        f.write(f"  {key}: {value}\n")
    f.write(f"\nTest metrics:\n")
    f.write(f"  Accuracy: {accuracy:.4f}\n")
    f.write(f"  F1-Score (weighted): {f1_weighted:.4f}\n")
    f.write(f"  F1-Score (macro): {f1_macro:.4f}\n")

print(f"✓ Hiperparámetros guardados: {params_path}")

✓ Modelo guardado: ..\models\xgb_model.joblib
✓ Label encoder guardado: ..\models\xgb_label_encoder.joblib
✓ Hiperparámetros guardados: ..\results\xgb_best_params.txt


## 11. Validación de Carga

In [15]:
# Verificar que el modelo se puede recargar correctamente
print("Validando recarga del modelo...")
start = time.time()

xgb_reload = joblib.load(model_path)
le_reload = joblib.load(le_path)
y_pred_reload = xgb_reload.predict(X_test[:100])  # Probar con 100 samples

load_time = time.time() - start

print(f"✓ Modelo recargado y probado en {load_time:.2f} segundos")
print(f"  Predicciones (encoded): {y_pred_reload[:5]}")
print(f"  Predicciones (decoded): {le_reload.inverse_transform(y_pred_reload[:5])}")
print(f"\n✅ Modelo XGBoost listo para uso")

Validando recarga del modelo...
✓ Modelo recargado y probado en 0.11 segundos
  Predicciones (encoded): [2 2 1 4 3]
  Predicciones (decoded): ['joy' 'joy' 'fear' 'sadness' 'love']

✅ Modelo XGBoost listo para uso


## Resumen

- **Modelo**: XGBoost multi-class con sample weights (balanced)
- **Optimización**: Optuna 50 trials en subset de 10K samples
- **Dataset**: {"Subset 40K train" if USE_SUBSET else "Full 436K train"}
- **Métricas test**: Ver sección 8
- **Archivos generados**:
  - `models/xgb_model{suffix}.joblib`
  - `models/xgb_label_encoder{suffix}.joblib`
  - `results/xgb_best_params{suffix}.txt`

**Siguiente paso**: Entrenar KNN (notebook 08) y comparar resultados.